# MSMARCO-XI — Standalone Full-Corpus FAISS Indexer
## Local machine — no Google Drive / Colab

This notebook is intentionally **standalone**. You do **not** need the RAG project ZIP.

It is designed around a modest local machine's memory limits:

```text
ONE target language
      ↓
download ONE source Parquet file
      ↓
PyArrow batches of 64 rows
      ↓
disk-backed LMDB passage dedupe
      ↓
production multi-view chunking
      ↓
multilingual-e5-small, FP16, batch 32
      ↓
append embeddings.f16 directly to disk
      ↓
append mmap payload files directly to disk
      ↓
delete source Parquet
      ↓
FAISS IVF-PQ from memory-mapped FP16 vectors
      ↓
archive directly to a local output directory
      ↓
delete local target workspace
      ↓
NEXT LANGUAGE
```

### Output compatibility

Each completed archive contains the same runtime layout expected by the optimized local RAG project:

```text
hi/
├── index.faiss
├── embeddings.f16
├── manifest.json
├── vector_manifest.json
└── payload/
    ├── texts.bin
    ├── offsets.u64
    ├── parent_hashes.u64
    ├── families.u8
    ├── selected.u8
    └── manifest.json
```

### Languages

`hi, as, bn, gu, kn, ml, mr, ne, or, pa, sa, ta, te, ur`

English is built once from the original English passages in the Hindi MSMARCO-XI files.

### Resume model

A completed target is immediately saved to the local output directory with a `.complete.json` marker.  
If the run is interrupted, rerun the notebook. Completed targets are skipped.

A target that did not finish is rebuilt from scratch. This is deliberate: it is much safer than trying to recover partially coordinated LMDB + vector files after a VM crash.

## 1. Local workspace (no Drive mount)

In [1]:
# Local mode: no Google Drive. Output paths are configured in the
# Configuration cell (OUTPUT_ROOT / LOCAL_ROOT).
print("Local mode — no Drive mount needed.")


Local mode — no Drive mount needed.


## 2. Configuration

The defaults below are intentionally conservative for a **local GPU machine with a ~5 GB system RAM budget**.

You can run the notebook as-is.

If you want a short verification first, set:

```python
SMOKE_TEST = True
```

After it succeeds, set it back to `False` and rerun.

In [2]:
from pathlib import Path
import os

# -------------------- main switches --------------------

SMOKE_TEST = False
SMOKE_ROWS_PER_FILE = 2_000

# Recommended: Build Hindi first for fast validation (~3-5 mins), then add other Indic languages
TARGETS = [
    "hi",
    # "en",
    # "as",
    # "bn",
    # "gu",
    # "kn",
    # "ml",
    # "mr",
    # "ne",
    # "or",
    # "pa",
    # "sa",
    # "ta",
    # "te",
    # "ur",
]

# Re-running the notebook skips targets already safely archived in the local output directory.
SKIP_COMPLETED = True

# -------------------- paths (Portable across Windows & Linux) --------------------

BASE_DIR = Path(".").resolve()
OUTPUT_ROOT = BASE_DIR / "data" / "production" / "indexes"
LOCAL_ROOT = BASE_DIR / "data" / "production" / "work"
SOURCE_ROOT = LOCAL_ROOT / "source"
WORK_ROOT = LOCAL_ROOT / "work"

# -------------------- embedding --------------------

EMBEDDING_MODEL = "intfloat/multilingual-e5-small"
EMBEDDING_DIM = 384

# High-throughput batching optimized for GPU/CPU acceleration
PARQUET_ROW_BATCH = 1024
EMBEDDING_BATCH = 128
MODEL_MAX_SEQ_LENGTH = 128

# -------------------- FAISS --------------------

PQ_M = 48
PQ_NBITS = 4
NPROBE = 8

# Training allocation: 40k * 384 * float32 ~= 60 MiB raw.
FAISS_TRAIN_SAMPLE = 40_000

# Batch add to FAISS index:
FAISS_ADD_BATCH = 50_000

# -------------------- chunking --------------------

MIN_SENTENCE_WINDOW_WORDS = 96
ADAPTIVE_MAX_WORDS = 320
SENTENCE_WINDOW_SIZE = 3

# -------------------- dedupe --------------------

# LMDB map_size is address-space capacity; it does not allocate 32 GiB RAM.
LMDB_MAP_GB = 32

# -------------------------------------------------------

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Local output:", OUTPUT_ROOT)
print("Targets:", TARGETS)
print("Smoke test:", SMOKE_TEST)


Local output: /home/brajesh_kurkure/study/TSMF/msmarco_xi_indexes
Targets: ['hi', 'en', 'as', 'bn', 'gu', 'kn', 'ml', 'mr', 'ne', 'or', 'pa', 'sa', 'ta', 'te', 'ur']
Smoke test: False


## 3. Configure the runtime before loading ML libraries

This disables optional TensorFlow/JAX initialization and limits CPU thread pools.

In [3]:
import os
import torch

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

num_cpus = str(min(8, os.cpu_count() or 4))
os.environ["OMP_NUM_THREADS"] = num_cpus
os.environ["MKL_NUM_THREADS"] = num_cpus
os.environ["OPENBLAS_NUM_THREADS"] = num_cpus
os.environ["NUMEXPR_MAX_THREADS"] = num_cpus
os.environ["TOKENIZERS_PARALLELISM"] = "true"

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")

# Helps reduce CUDA allocator fragmentation.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

print(f"Environment configured with {num_cpus} threads. CUDA available: {torch.cuda.is_available()}")


Low-memory environment configured.


## 4. Install only the required packages

The notebook does **not** use the Hugging Face `datasets` library for corpus processing.

In [4]:
import subprocess, sys

# zstd is expected to be installed system-wide.
# If it is missing, install it with your OS package manager, e.g.:  sudo apt install zstd

packages = [
    "huggingface_hub[hf_xet]>=0.27.0",
    "pyarrow>=17.0.0",
    "sentence-transformers>=3.2.0",
    "faiss-cpu>=1.9.0",
    "lmdb>=1.5.1",
    "psutil>=6.0.0",
    "zstandard>=0.23.0",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

print("Dependencies installed.")

Dependencies installed.


## 5. Optional Hugging Face token

For a large download, set `HF_TOKEN` in your environment (or log in once with `huggingface-cli login`).

The dataset is public, so the notebook still works without it, but an authenticated token can reduce rate-limit problems.

In [5]:
import os

# Local mode: read HF_TOKEN from the environment. If it is not set,
# huggingface_hub falls back to the cached huggingface-cli token.
HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF token loaded from environment.")
else:
    print("No HF_TOKEN in environment — continuing with public access.")


No HF_TOKEN in environment — continuing with public access.


## 6. Check GPU, system RAM and local disk

In [6]:
import platform, shutil, sys, psutil, torch

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

vm = psutil.virtual_memory()
print(f"System RAM: {vm.total / 1024**3:.2f} GiB")
print(f"Available RAM: {vm.available / 1024**3:.2f} GiB")

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU VRAM: {props.total_memory / 1024**3:.2f} GiB")
else:
    print(
        "No CUDA GPU found — the model will run on CPU (slower)."
    )

du = shutil.disk_usage(LOCAL_ROOT)
print(f"Local disk total: {du.total / 1024**3:.1f} GiB")
print(f"Local disk free: {du.free / 1024**3:.1f} GiB")

Python: 3.12.3
Platform: Linux-7.0.0-28-generic-x86_64-with-glibc2.39
System RAM: 7.44 GiB
Available RAM: 1.39 GiB
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU VRAM: 3.68 GiB
Local disk total: 467.9 GiB
Local disk free: 255.1 GiB


## 7. Discover exactly which MSMARCO-XI files currently exist

The repository is checked dynamically instead of assuming every language has both train and validation files.

In [7]:
from huggingface_hub import HfApi

REPO_ID = "ai4bharat/MSMARCO-XI"

PREFIX = {
    "as": "asm",
    "bn": "ben",
    "gu": "guj",
    "hi": "hin",
    "kn": "kan",
    "ml": "mal",
    "mr": "mar",
    "ne": "nep",
    "or": "ori",
    "pa": "pan",
    "sa": "san",
    "ta": "tam",
    "te": "tel",
    "ur": "urd",
}

api = HfApi(token=HF_TOKEN)
REPO_FILES = set(
    api.list_repo_files(
        repo_id=REPO_ID,
        repo_type="dataset",
    )
)

def source_language_for_target(target):
    # English is built once from the English passage field in Hindi files.
    return "hi" if target == "en" else target

def available_source_files(target):
    lang = source_language_for_target(target)
    prefix = PREFIX[lang]

    candidates = [
        ("train", f"train/{prefix}train.parquet"),
        ("validation", f"validation/{prefix}val.parquet"),
    ]

    return [
        (split, filename)
        for split, filename in candidates
        if filename in REPO_FILES
    ]

AVAILABLE_FILES = {}

for target in TARGETS:
    files = available_source_files(target)
    AVAILABLE_FILES[target] = files

    if not files:
        print(f"{target:>2}: no source file available")
    else:
        print(
            f"{target:>2}: "
            + ", ".join(f"{split}={name}" for split, name in files)
        )

/home/brajesh_kurkure/study/TSMF/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


hi: train=train/hintrain.parquet, validation=validation/hinval.parquet
en: train=train/hintrain.parquet, validation=validation/hinval.parquet
as: train=train/asmtrain.parquet, validation=validation/asmval.parquet
bn: train=train/bentrain.parquet, validation=validation/benval.parquet
gu: train=train/gujtrain.parquet, validation=validation/gujval.parquet
kn: train=train/kantrain.parquet, validation=validation/kanval.parquet
ml: train=train/maltrain.parquet, validation=validation/malval.parquet
mr: train=train/martrain.parquet, validation=validation/marval.parquet
ne: train=train/neptrain.parquet, validation=validation/nepval.parquet
or: train=train/oritrain.parquet, validation=validation/orival.parquet
pa: train=train/pantrain.parquet, validation=validation/panval.parquet
sa: train=train/santrain.parquet, validation=validation/sanval.parquet
ta: train=train/tamtrain.parquet, validation=validation/tamval.parquet
te: validation=validation/telval.parquet
ur: train=train/urdtrain.parquet, va

## 8. Core low-memory utilities

This cell defines:

- RAM / disk monitoring
- atomic JSON writes
- text normalization
- production multi-view chunking
- mmap payload writer
- LMDB dedupe store
- safe source-file downloader

In [8]:
import gc
import hashlib
import json
import math
import os
import re
import shutil
import time
from pathlib import Path

import lmdb
import numpy as np
import psutil
import torch
from huggingface_hub import hf_hub_download


FAMILY_TO_CODE = {
    "native_passage": 0,
    "sentence_window": 1,
    "adaptive": 2,
}


def release_memory():
    gc.collect()

    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def ram_report(prefix=""):
    vm = psutil.virtual_memory()
    print(
        f"{prefix}RAM used={vm.used/1024**3:.2f} GiB | "
        f"available={vm.available/1024**3:.2f} GiB"
    )


def disk_report(prefix=""):
    du = shutil.disk_usage(LOCAL_ROOT)
    print(
        f"{prefix}SSD used={du.used/1024**3:.1f} GiB | "
        f"free={du.free/1024**3:.1f} GiB"
    )


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = Path(str(path) + ".tmp")
    tmp.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    os.replace(tmp, path)


def normalize_text(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def passage_key(text):
    # 16-byte key keeps the LMDB entry compact.
    normalized = normalize_text(text).casefold().encode("utf-8")
    return hashlib.blake2b(normalized, digest_size=16).digest()


def parent_hash(text, language):
    payload = (
        language + "\0" + normalize_text(text).casefold()
    ).encode("utf-8")

    return int.from_bytes(
        hashlib.blake2b(payload, digest_size=8).digest(),
        "little",
    )


def split_sentences(text):
    text = normalize_text(text)

    if not text:
        return []

    pieces = re.split(r"(?<=[.!?।॥])\s+", text)

    return [
        piece.strip()
        for piece in pieces
        if piece and piece.strip()
    ]


def production_chunk_views(text):
    # Mirrors the production policy used by the local RAG project:
    # native passage always, sentence windows only for structured passages,
    # adaptive chunks only for genuinely long passages.
    text = normalize_text(text)

    if not text:
        return []

    candidates = [("native_passage", text)]

    words = text.split()
    sentences = split_sentences(text)

    if (
        len(sentences) >= 4
        and len(words) >= MIN_SENTENCE_WINDOW_WORDS
    ):
        width = SENTENCE_WINDOW_SIZE
        stride = max(1, width - 1)

        for start in range(0, len(sentences), stride):
            window = sentences[start : start + width]

            if window:
                candidates.append(
                    ("sentence_window", " ".join(window))
                )

            if start + width >= len(sentences):
                break

    if len(words) > ADAPTIVE_MAX_WORDS:
        current = []
        count = 0

        adaptive = []

        for sentence in sentences:
            n = len(sentence.split())

            if (
                current
                and count + n > ADAPTIVE_MAX_WORDS
            ):
                adaptive.append(" ".join(current))

                # one-sentence boundary overlap
                current = [current[-1], sentence]
                count = sum(
                    len(item.split())
                    for item in current
                )
            else:
                current.append(sentence)
                count += n

        if current:
            adaptive.append(" ".join(current))

        for piece in adaptive:
            candidates.append(("adaptive", piece))

    # Remove identical views.
    seen = set()
    output = []

    for family, chunk_text in candidates:
        chunk_text = normalize_text(chunk_text)
        key = chunk_text.casefold()

        if not chunk_text or key in seen:
            continue

        seen.add(key)
        output.append(
            {
                "text": chunk_text,
                "chunk_family": family,
            }
        )

    return output


class PayloadWriter:
    def __init__(self, root, language):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

        self.language = language

        self.text_path = self.root / "texts.bin"
        self.offset_path = self.root / "offsets.u64"
        self.parent_path = self.root / "parent_hashes.u64"
        self.family_path = self.root / "families.u8"
        self.selected_path = self.root / "selected.u8"
        self.manifest_path = self.root / "manifest.json"

        self.text_handle = open(self.text_path, "wb")
        self.offset_handle = open(self.offset_path, "wb")
        self.parent_handle = open(self.parent_path, "wb")
        self.family_handle = open(self.family_path, "wb")
        self.selected_handle = open(self.selected_path, "wb")

        self.count = 0
        self.text_bytes = 0

    def append_batch(self, docs):
        if not docs:
            return

        n = len(docs)

        offsets = np.empty(n, dtype="<u8")
        parents = np.empty(n, dtype="<u8")
        families = np.empty(n, dtype=np.uint8)
        selected = np.empty(n, dtype=np.uint8)

        cursor = self.text_bytes
        encoded = []

        for i, doc in enumerate(docs):
            raw = doc["text"].encode("utf-8")
            encoded.append(raw)

            offsets[i] = cursor
            cursor += len(raw)

            parents[i] = np.uint64(doc["parent_hash"])
            families[i] = FAMILY_TO_CODE.get(
                doc["chunk_family"],
                255,
            )
            selected[i] = (
                1 if doc.get("is_selected", False) else 0
            )

        self.offset_handle.write(offsets.tobytes())
        self.parent_handle.write(parents.tobytes())
        self.family_handle.write(families.tobytes())
        self.selected_handle.write(selected.tobytes())

        for raw in encoded:
            self.text_handle.write(raw)

        self.count += n
        self.text_bytes = cursor

    def flush(self, fsync=False):
        handles = [
            self.text_handle,
            self.offset_handle,
            self.parent_handle,
            self.family_handle,
            self.selected_handle,
        ]

        for handle in handles:
            handle.flush()

            if fsync:
                os.fsync(handle.fileno())

    def finalize(self):
        # Final sentinel makes offsets length = count + 1.
        self.offset_handle.write(
            np.asarray(
                [self.text_bytes],
                dtype="<u8",
            ).tobytes()
        )

        self.flush(fsync=True)

        manifest = {
            "version": 1,
            "backend": "mmap_payload_v1",
            "language": self.language,
            "count": int(self.count),
            "text_bytes": int(self.text_bytes),
            "finalized": True,
            "files": {
                "texts": "texts.bin",
                "offsets": "offsets.u64",
                "parent_hashes": "parent_hashes.u64",
                "families": "families.u8",
                "selected": "selected.u8",
            },
        }

        atomic_json(
            self.manifest_path,
            manifest,
        )

        return manifest

    def close(self):
        for handle in [
            self.text_handle,
            self.offset_handle,
            self.parent_handle,
            self.family_handle,
            self.selected_handle,
        ]:
            try:
                handle.close()
            except Exception:
                pass


class DedupeStore:
    def __init__(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        self.env = lmdb.open(
            str(path),
            map_size=LMDB_MAP_GB * 1024**3,
            subdir=False,
            lock=True,
            readahead=False,
            meminit=False,
            map_async=True,
            max_dbs=1,
        )

    def contains(self, key):
        with self.env.begin(write=False) as txn:
            return txn.get(key) is not None

    def add_many(self, keys):
        if not keys:
            return

        with self.env.begin(write=True) as txn:
            for key in keys:
                txn.put(
                    key,
                    b"1",
                    overwrite=False,
                )

    def sync(self):
        self.env.sync()

    def close(self):
        try:
            self.env.sync()
        finally:
            self.env.close()


def download_source(filename):
    # Put only the current source Parquet on local disk.
    target_root = SOURCE_ROOT / "repo"
    target_root.mkdir(parents=True, exist_ok=True)

    for attempt in range(1, 6):
        try:
            print(
                f"Downloading {filename} "
                f"(attempt {attempt}/5)..."
            )

            path = hf_hub_download(
                repo_id=REPO_ID,
                repo_type="dataset",
                filename=filename,
                token=HF_TOKEN,
                local_dir=str(target_root),
            )

            return Path(path)

        except Exception as exc:
            print("Download error:", repr(exc))

            if attempt == 5:
                raise

            time.sleep(5 * attempt)


ram_report()
disk_report()

RAM used=6.02 GiB | available=1.43 GiB
SSD used=188.9 GiB | free=255.1 GiB


## 9. Load the multilingual E5 model once on the GPU

The model itself stays on the GPU. Corpus data stays on CPU / disk.

We use FP16 model weights and cap the transformer sequence length at 256 for predictable memory use.

In [9]:
from sentence_transformers import SentenceTransformer

print("Loading:", EMBEDDING_MODEL)

MODEL = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

MODEL.max_seq_length = MODEL_MAX_SEQ_LENGTH
MODEL.half()
MODEL.eval()

actual_dim = MODEL.get_sentence_embedding_dimension()

if int(actual_dim) != EMBEDDING_DIM:
    EMBEDDING_DIM = int(actual_dim)
    print("Embedding dimension updated to:", EMBEDDING_DIM)

# Small warmup.
_ = MODEL.encode(
    ["query: warmup"],
    batch_size=1,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

release_memory()

print(
    "Model ready | dim=",
    EMBEDDING_DIM,
    "| max_seq_length=",
    MODEL.max_seq_length,
)
ram_report()

Loading: intfloat/multilingual-e5-small


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5645.28it/s]
/tmp/ipykernel_459749/2662261074.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  actual_dim = MODEL.get_sentence_embedding_dimension()


Model ready | dim= 384 | max_seq_length= 128
RAM used=6.47 GiB | available=0.97 GiB


## 10. Direct Parquet → embeddings + payload encoder

There is **no intermediate processed Parquet corpus**.

This is the main memory-saving change.

In [10]:
import pyarrow.parquet as pq


def target_workdir(target):
    return WORK_ROOT / target


def artifact_dir(target):
    return target_workdir(target) / target


def target_archive(target):
    return OUTPUT_ROOT / f"msmarco_xi_{target}_runtime_local.tar.zst"


def target_marker(target):
    return OUTPUT_ROOT / f"{target}.complete.json"


def target_complete(target):
    return (
        target_archive(target).exists()
        and target_marker(target).exists()
    )


def encode_text_batch(texts, batch_size):
    prepared = [
        text
        if text.startswith("passage: ")
        else "passage: " + text
        for text in texts
    ]

    with torch.inference_mode():
        matrix = MODEL.encode(
        prepared,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
            show_progress_bar=False,
        )

    return np.ascontiguousarray(
        np.asarray(
            matrix,
            dtype=np.float32,
        )
    )


def process_target(target):
    if SKIP_COMPLETED and target_complete(target):
        print(
            f"\n⏭️ {target.upper()} already completed — skipping."
        )
        return json.loads(
            target_marker(target).read_text(
                encoding="utf-8"
            )
        )

    source_lang = source_language_for_target(target)
    source_files = AVAILABLE_FILES[target]

    if not source_files:
        raise RuntimeError(
            f"No available MSMARCO-XI source files for {target}"
        )

    work = target_workdir(target)

    if work.exists():
        # A previous incomplete target is rebuilt from scratch.
        shutil.rmtree(work)

    art = artifact_dir(target)
    payload_root = art / "payload"

    art.mkdir(parents=True, exist_ok=True)

    embeddings_path = art / "embeddings.f16"
    vector_manifest_path = art / "vector_manifest.json"

    dedupe = DedupeStore(
        work / "dedupe.lmdb"
    )

    payload = PayloadWriter(
        payload_root,
        language=target,
    )

    embedding_handle = open(
        embeddings_path,
        "wb",
    )

    docs_seen = 0
    unique_passages = 0
    vectors_written = 0
    started = time.perf_counter()

    # Buffer contains whole-passage views. A passage is never split across two
    # flushes, which keeps dedupe behavior simple.
    chunk_buffer = []
    passage_keys_waiting_commit = []
    pending_passage_keys = set()

    current_batch = EMBEDDING_BATCH

    def flush_chunks():
        nonlocal chunk_buffer
        nonlocal passage_keys_waiting_commit
        nonlocal pending_passage_keys
        nonlocal vectors_written
        nonlocal current_batch

        if not chunk_buffer:
            return

        while True:
            try:
                matrix = encode_text_batch(
                    [item["text"] for item in chunk_buffer],
                    batch_size=current_batch,
                )
                break

            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                gc.collect()

                new_batch = max(
                    8,
                    current_batch // 2,
                )

                if new_batch == current_batch:
                    raise

                current_batch = new_batch

                print(
                    "CUDA OOM recovered. "
                    f"New embedding batch={current_batch}"
                )

        np.asarray(
            matrix,
            dtype="<f2",
        ).tofile(embedding_handle)

        payload.append_batch(
            chunk_buffer
        )

        # Mark passage hashes only after vector/payload writes succeed.
        dedupe.add_many(
            passage_keys_waiting_commit
        )

        vectors_written += len(chunk_buffer)

        del matrix
        chunk_buffer = []
        passage_keys_waiting_commit = []
        pending_passage_keys = set()

    for split, filename in source_files:
        print("\n" + "=" * 100)
        print(
            f"{target.upper()} | "
            f"{split} | {filename}"
        )
        print("=" * 100)

        source_path = download_source(
            filename
        )

        disk_report("After download: ")

        pf = pq.ParquetFile(
            source_path,
            memory_map=True,
        )

        print(
            "Parquet rows:",
            f"{pf.metadata.num_rows:,}",
            "| row groups:",
            pf.metadata.num_row_groups,
        )

        processed_rows_this_file = 0

        columns = [
            "query_id",
            "passages",
        ]

        for record_batch in pf.iter_batches(
            batch_size=PARQUET_ROW_BATCH,
            columns=columns,
            use_threads=False,
        ):
            rows = record_batch.to_pylist()
            batch_seen_keys = set()

            # One LMDB read transaction per Arrow batch instead of one
            # transaction per passage. This is much faster for millions of rows.
            with dedupe.env.begin(write=False) as read_txn:
                for row in rows:
                    docs_seen += 1
                    processed_rows_this_file += 1
    
                    passage_struct = (
                        row.get("passages") or {}
                    )
    
                    selected_flags = (
                        passage_struct.get("is_selected")
                        or []
                    )
    
                    if target == "en":
                        texts = (
                            passage_struct.get(
                                "English_passages"
                            )
                            or []
                        )
                    else:
                        texts = (
                            passage_struct.get(
                                "Translated_passages"
                            )
                            or []
                        )
    
                    for passage_index, raw_text in enumerate(texts):
                        text = normalize_text(
                            raw_text
                        )
    
                        if not text:
                            continue
    
                        key = passage_key(
                            text
                        )
    
                        if (
                            read_txn.get(key) is not None
                            or key in pending_passage_keys
                            or key in batch_seen_keys
                        ):
                            continue
    
                        views = production_chunk_views(
                            text
                        )
    
                        if not views:
                            continue
    
                        # Flush before adding the next passage if it would exceed
                        # our bounded embedding buffer.
                        if (
                            chunk_buffer
                            and len(chunk_buffer) + len(views)
                            > max(EMBEDDING_BATCH, current_batch)
                        ):
                            flush_chunks()
    
                        selected = (
                            bool(selected_flags[passage_index])
                            if passage_index < len(selected_flags)
                            else False
                        )
    
                        p_hash = parent_hash(
                            text,
                            target,
                        )
    
                        for view in views:
                            chunk_buffer.append(
                                {
                                    "text": view["text"],
                                    "chunk_family": view["chunk_family"],
                                    "parent_hash": p_hash,
                                    "is_selected": selected,
                                }
                            )
    
                        passage_keys_waiting_commit.append(
                            key
                        )
                        pending_passage_keys.add(key)
                        batch_seen_keys.add(key)
    
                        unique_passages += 1
    
                        if len(chunk_buffer) >= current_batch:
                            flush_chunks()
    
            del rows
            del record_batch

            if (
                SMOKE_TEST
                and processed_rows_this_file >= SMOKE_ROWS_PER_FILE
            ):
                print(
                    "Smoke row limit reached:",
                    SMOKE_ROWS_PER_FILE,
                )
                break

            if docs_seen % 10_000 < PARQUET_ROW_BATCH:
                embedding_handle.flush()
                payload.flush()
                dedupe.sync()

                print(
                    f"rows={docs_seen:,} | "
                    f"unique passages={unique_passages:,} | "
                    f"vectors={vectors_written:,} | "
                    f"embed_batch={current_batch}"
                )

                ram_report("  ")
                disk_report("  ")

        flush_chunks()

        embedding_handle.flush()
        os.fsync(
            embedding_handle.fileno()
        )

        payload.flush(fsync=True)
        dedupe.sync()

        # Release the Parquet reader before deleting the local source file.
        del pf
        release_memory()

        try:
            source_path.unlink()
        except Exception:
            pass

        # Remove empty source directories when possible.
        try:
            parent = source_path.parent

            while (
                parent != SOURCE_ROOT
                and parent.exists()
                and not any(parent.iterdir())
            ):
                parent.rmdir()
                parent = parent.parent
        except Exception:
            pass

        release_memory()
        disk_report("After source cleanup: ")

    flush_chunks()

    payload_manifest = payload.finalize()

    payload.close()
    dedupe.close()
    embedding_handle.close()

    elapsed = time.perf_counter() - started

    vector_manifest = {
        "status": "success",
        "version": 1,
        "language": target,
        "dimension": EMBEDDING_DIM,
        "documents": docs_seen,
        "unique_passages": unique_passages,
        "vectors": vectors_written,
        "strategy": "multi_view",
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dtype": "float16",
        "embedding_file": str(
            embeddings_path
        ),
        "embedding_bytes": embeddings_path.stat().st_size,
        "payload": payload_manifest,
        "source_files": [
            filename
            for _, filename in source_files
        ],
        "elapsed_seconds": round(
            elapsed,
            3,
        ),
    }

    atomic_json(
        vector_manifest_path,
        vector_manifest,
    )

    # Dedupe DB is not needed after vectorization.
    dedupe_path = work / "dedupe.lmdb"

    if dedupe_path.exists():
        dedupe_path.unlink()

    release_memory()

    print(
        f"\n✅ Vectorization complete for {target}: "
        f"{vectors_written:,} vectors"
    )

    ram_report()
    disk_report()

    return vector_manifest

## 11. Low-RAM FAISS IVF-PQ builder

The FP16 embedding matrix remains memory-mapped on disk.

Only the training sample and a 2,000-vector add batch are converted to float32.

In [11]:
import faiss


def choose_nlist(total_vectors):
    if total_vectors < 50_000:
        return 0

    # Conservative IVF count for a 16 GB build.
    target = int(math.sqrt(total_vectors))

    target = max(
        256,
        min(
            4096,
            target,
        ),
    )

    power = 1 << int(
        round(
            math.log2(target)
        )
    )

    return int(
        max(
            256,
            min(
                4096,
                power,
            ),
        )
    )


def build_faiss_for_target(target):
    art = artifact_dir(target)

    vector_manifest = json.loads(
        (art / "vector_manifest.json").read_text(
            encoding="utf-8"
        )
    )

    total = int(
        vector_manifest["vectors"]
    )

    dim = int(
        vector_manifest["dimension"]
    )

    if total <= 0:
        raise RuntimeError(
            f"No vectors available for {target}"
        )

    emb_path = art / "embeddings.f16"
    index_path = art / "index.faiss"
    manifest_path = art / "manifest.json"

    raw = np.memmap(
        emb_path,
        dtype="<f2",
        mode="r",
        shape=(total, dim),
    )

    nlist = choose_nlist(
        total
    )

    if nlist == 0:
        index = faiss.IndexFlatIP(
            dim
        )
        index_type = "FlatIP"

    else:
        # Prefer 4-bit FastScan. Fall back automatically if the wheel does not
        # expose that factory on this machine.
        factory = (
            f"IVF{nlist},"
            f"PQ{PQ_M}x4fsr"
        )

        try:
            index = faiss.index_factory(
                dim,
                factory,
                faiss.METRIC_INNER_PRODUCT,
            )

            index_type = factory

        except Exception as exc:
            print(
                "FastScan factory unavailable:",
                repr(exc),
            )
            print(
                "Falling back to standard IVF-PQ."
            )

            quantizer = faiss.IndexFlatIP(
                dim
            )

            index = faiss.IndexIVFPQ(
                quantizer,
                dim,
                nlist,
                PQ_M,
                max(
                    4,
                    PQ_NBITS,
                ),
                faiss.METRIC_INNER_PRODUCT,
            )

            index_type = (
                f"IVF{nlist}-"
                f"PQ{PQ_M}x{max(4, PQ_NBITS)}"
            )

        if not index.is_trained:
            # FAISS generally wants many examples per centroid. We use at least
            # 40*nlist where possible, while capping RAM at 50k samples.
            sample_n = min(
                total,
                max(
                    FAISS_TRAIN_SAMPLE,
                    min(
                        50_000,
                        nlist * 40,
                    ),
                ),
            )

            print(
                f"Training {index_type} "
                f"with {sample_n:,} vectors..."
            )

            rng = np.random.default_rng(
                42
            )

            ids = rng.choice(
                total,
                size=sample_n,
                replace=False,
            )

            sample = np.asarray(
                raw[ids],
                dtype=np.float32,
            )

            faiss.normalize_L2(
                sample
            )

            index.train(
                np.ascontiguousarray(
                    sample
                )
            )

            del sample
            del ids

            release_memory()

    if hasattr(index, "nprobe"):
        index.nprobe = NPROBE

    started = time.perf_counter()

    for lo in range(
        0,
        total,
        FAISS_ADD_BATCH,
    ):
        hi = min(
            total,
            lo + FAISS_ADD_BATCH,
        )

        batch = np.asarray(
            raw[lo:hi],
            dtype=np.float32,
        )

        faiss.normalize_L2(
            batch
        )

        index.add(
            np.ascontiguousarray(
                batch
            )
        )

        del batch

        if (
            lo == 0
            or hi == total
            or hi % 250_000 < FAISS_ADD_BATCH
        ):
            print(
                f"FAISS add: "
                f"{hi:,}/{total:,}"
            )
            ram_report("  ")

    if hasattr(index, "nprobe"):
        index.nprobe = NPROBE

    faiss.write_index(
        index,
        str(index_path),
    )

    build_seconds = (
        time.perf_counter() - started
    )

    manifest = {
        "status": "success",
        "version": 1,
        "production": True,
        "language": target,
        "vectors": total,
        "dimension": dim,
        "index_type": index_type,
        "nprobe": int(
            getattr(
                index,
                "nprobe",
                0,
            )
        ),
        "index_file": str(
            index_path
        ),
        "index_bytes": index_path.stat().st_size,
        "refinement_file": str(
            emb_path
        ),
        "payload_root": str(
            art / "payload"
        ),
        "build_seconds": round(
            build_seconds,
            3,
        ),
    }

    atomic_json(
        manifest_path,
        manifest,
    )

    # Explicitly release potentially large FAISS inverted lists before archive.
    del index
    del raw

    release_memory()

    print(
        f"✅ FAISS built for {target}: "
        f"{index_type}"
    )

    ram_report()
    disk_report()

    return manifest

## 12. Validate + archive a completed target to the local output directory

Validation uses file sizes/manifests and does **not** reload the full FAISS index into RAM.

In [12]:
import subprocess


def validate_artifact(target):
    art = artifact_dir(target)

    vector_manifest = json.loads(
        (art / "vector_manifest.json").read_text(
            encoding="utf-8"
        )
    )

    index_manifest = json.loads(
        (art / "manifest.json").read_text(
            encoding="utf-8"
        )
    )

    payload_manifest = json.loads(
        (art / "payload" / "manifest.json").read_text(
            encoding="utf-8"
        )
    )

    n = int(
        vector_manifest["vectors"]
    )

    d = int(
        vector_manifest["dimension"]
    )

    emb = art / "embeddings.f16"
    index_file = art / "index.faiss"
    offsets = art / "payload" / "offsets.u64"
    parents = art / "payload" / "parent_hashes.u64"
    families = art / "payload" / "families.u8"
    selected = art / "payload" / "selected.u8"
    texts = art / "payload" / "texts.bin"

    checks = {
        "vector_count_positive":
            n > 0,
        "embedding_size":
            emb.exists()
            and emb.stat().st_size == n * d * 2,
        "index_nonempty":
            index_file.exists()
            and index_file.stat().st_size > 0,
        "payload_finalized":
            bool(
                payload_manifest.get(
                    "finalized"
                )
            ),
        "payload_count":
            int(
                payload_manifest["count"]
            ) == n,
        "offset_count":
            offsets.stat().st_size == (n + 1) * 8,
        "parent_count":
            parents.stat().st_size == n * 8,
        "families_count":
            families.stat().st_size == n,
        "selected_count":
            selected.stat().st_size == n,
        "texts_nonempty":
            texts.stat().st_size > 0,
    }

    report = {
        "target": target,
        "vectors": n,
        "dimension": d,
        "index_type": index_manifest["index_type"],
        "embedding_gib": round(
            emb.stat().st_size / 1024**3,
            3,
        ),
        "index_gib": round(
            index_file.stat().st_size / 1024**3,
            3,
        ),
        "checks": checks,
        "all_ok": all(
            checks.values()
        ),
    }

    print(
        json.dumps(
            report,
            indent=2,
            ensure_ascii=False,
        )
    )

    if not report["all_ok"]:
        raise RuntimeError(
            f"Artifact validation failed for {target}"
        )

    return report


def archive_target(target):
    report = validate_artifact(
        target
    )

    art_parent = target_workdir(
        target
    )

    final_archive = target_archive(
        target
    )

    partial_archive = Path(
        str(final_archive) + ".partial"
    )

    if partial_archive.exists():
        partial_archive.unlink()

    # Archive directly to the local output directory.
    cmd = [
        "tar",
        "-C",
        str(art_parent),
        "-I",
        "zstd -T1 -1",
        "-cf",
        str(partial_archive),
        target,
    ]

    print(
        "\n$",
        " ".join(cmd),
    )

    subprocess.run(
        cmd,
        check=True,
    )

    os.replace(
        partial_archive,
        final_archive,
    )

    marker = {
        "status": "complete",
        "target": target,
        "archive": str(
            final_archive
        ),
        "archive_bytes": final_archive.stat().st_size,
        "embedding_model": EMBEDDING_MODEL,
        "model_max_seq_length": MODEL_MAX_SEQ_LENGTH,
        "embedding_batch": EMBEDDING_BATCH,
        "parquet_row_batch": PARQUET_ROW_BATCH,
        "faiss_train_sample": FAISS_TRAIN_SAMPLE,
        "faiss_add_batch": FAISS_ADD_BATCH,
        "pq_m": PQ_M,
        "pq_nbits": PQ_NBITS,
        "nprobe": NPROBE,
        "validation": report,
        "completed_at": time.strftime(
            "%Y-%m-%dT%H:%M:%SZ",
            time.gmtime(),
        ),
    }

    atomic_json(
        target_marker(
            target
        ),
        marker,
    )

    print(
        f"\n✅ Saved locally: "
        f"{final_archive}"
    )

    print(
        "Archive size:",
        round(
            final_archive.stat().st_size / 1024**3,
            3,
        ),
        "GiB",
    )

    # The Drive copy + marker now define completion.
    shutil.rmtree(
        target_workdir(target),
        ignore_errors=True,
    )

    # Remove download cache/source residue, while keeping model cache.
    if SOURCE_ROOT.exists():
        shutil.rmtree(
            SOURCE_ROOT,
            ignore_errors=True,
        )

    SOURCE_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    release_memory()

    ram_report()
    disk_report()

    return marker

# 13. Build the entire currently available MSMARCO-XI corpus

This is the only long-running cell.

For each target:

```text
download → encode → FAISS → archive → cleanup
```

If the run is interrupted, rerun the notebook. Completed targets are skipped.

In [ ]:
RESULTS = {}

for target in TARGETS:
    if SKIP_COMPLETED and target_complete(target):
        print(
            "\n" + "=" * 100
        )
        print(
            f"⏭️ {target.upper()} already completed — skipping."
        )
        print(
            target_archive(target)
        )

        RESULTS[target] = {
            "status": "skipped_completed",
        }

        continue

    if not AVAILABLE_FILES.get(target):
        print(
            f"⚠️ {target.upper()} has no currently available upstream file. Skipping."
        )

        RESULTS[target] = {
            "status": "no_upstream_file",
        }

        continue

    print(
        "\n" + "#" * 100
    )
    print(
        f"# BUILDING TARGET: {target.upper()}"
    )
    print(
        "#" * 100
    )

    ram_report()
    disk_report()

    try:
        vector_manifest = process_target(
            target
        )

        index_manifest = build_faiss_for_target(
            target
        )

        marker = archive_target(
            target
        )

        RESULTS[target] = {
            "status": "complete",
            "vectors": vector_manifest["vectors"],
            "index_type": index_manifest["index_type"],
            "archive": marker["archive"],
        }

    except Exception as exc:
        print(
            "\n" + "!" * 100
        )
        print(
            f"❌ TARGET FAILED: {target.upper()}"
        )
        print(
            repr(exc)
        )
        print(
            "!" * 100
        )

        print(
            "\nCompleted targets in the local output directory are safe. "
            "The incomplete target can be rebuilt by rerunning the notebook."
        )

        raise

print(
    "\n🎉 Finished all currently available requested targets."
)


####################################################################################################
# BUILDING TARGET: HI
####################################################################################################
RAM used=6.48 GiB | available=0.96 GiB
SSD used=188.9 GiB | free=255.1 GiB

HI | train | train/hintrain.parquet


## 14. Final verification

In [ ]:
completed = []
missing = []

for target in TARGETS:
    if target_complete(target):
        completed.append(
            target
        )
    else:
        missing.append(
            target
        )

print("Completed:", completed)
print("Missing:", missing)

print("\nLocal artifacts:")

for target in completed:
    archive = target_archive(
        target
    )

    print(
        f"✅ {target:>2}  "
        f"{archive.stat().st_size / 1024**3:8.3f} GiB  "
        f"{archive.name}"
    )

if not missing:
    print(
        "\n🎉 All requested targets are safely archived in the local output directory."
    )
else:
    print(
        "\nRerun the notebook to continue the missing targets."
    )

# 15. Use the completed indexes locally

The generated archives are already on the local disk:

```text
/home/brajesh_kurkure/study/TSMF/msmarco_xi_indexes/
```

Inside the local optimized RAG project:

```bash
mkdir -p data/production/artifacts

for file in msmarco_xi_*_runtime_local.tar.zst; do
    tar -I zstd -xf "$file" -C data/production/artifacts
done
```

You will end with:

```text
data/production/artifacts/
├── as/
├── bn/
├── en/
├── gu/
├── hi/
├── kn/
├── ml/
├── mr/
├── ne/
├── or/
├── pa/
├── sa/
├── ta/
├── te/
└── ur/
```

Use the **same query embedding model** locally:

```env
EMBEDDING_MODEL=intfloat/multilingual-e5-small
```

For a low-RAM local runtime, lazy-load language indexes rather than preloading every language.

---

## If the local GPU still reports CUDA OOM

The notebook already catches CUDA OOM during embedding and halves the current embedding batch automatically.

For an even more conservative starting point, change:

```python
EMBEDDING_BATCH = 8
PARQUET_ROW_BATCH = 16
```

System-RAM OOMs should be much less likely in this direct encoder because there is no intermediate processed corpus and only one 64-row Parquet batch is converted to Python at a time.

## Notes on "without any error"

No notebook can guarantee that a long-running job will never be interrupted by a power loss, a network error, or an OS event.

This notebook is designed so those platform-level events do not destroy already completed target indexes:

- each completed target is archived immediately;
- a completion JSON marker is written only after the archive finishes;
- rerunning skips completed targets;
- incomplete targets are rebuilt cleanly;
- no full 55.6 GB corpus is held in RAM or local disk at once.